In [351]:
import pandas as pd

In [353]:
table_dxsum = pd.read_csv('./tables/DXSUM.csv', low_memory=False)
table_adas = pd.read_csv('./tables/ADAS.csv', low_memory=False)
table_apoeres = pd.read_csv('./tables/APOERES.csv', low_memory=False)
table_ecogpt = pd.read_csv('./tables/ECOGPT.csv', low_memory=False)
table_ecogsp = pd.read_csv('./tables/ECOGSP.csv', low_memory=False)
table_neurobat = pd.read_csv('./tables/NEUROBAT.csv', low_memory=False)
table_pacc = pd.read_csv('./tables/PACC.csv', low_memory=False)
table_pibpetsurv = pd.read_csv('./tables/PIBPETSUVR.csv', low_memory=False)
table_ptdemog = pd.read_csv('./tables/PTDEMOG.csv', low_memory=False) #
table_ucsffsx7 = pd.read_csv('./tables/UCSFFSX7.csv', low_memory=False)
table_upennbiomkmaster = pd.read_csv('./tables/UPENNBIOMK_MASTER.csv', low_memory=False)

In [354]:
columns_dxsum            = ['RID', 'COLPROT', 'ORIGPROPT', 'PTID', 'SITEID', 'VISCODE', 'EXAMDATE', 'DIAGNOSIS']
columns_adas             = ['TOTSCORE']
columns_ptdemog          = ['PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY']
columns_apores           = ['GENOTYPE']
columns_ecogpt           = ['EcogPtMem', 'EcogPtLang', 'EcogPtVisspat', 'EcogPtPlan', 'EcogPtOrgan', 'EcogPtDivatt', 'EcogPtTotal']
columns_ecogsp           = ['EcogSPMem', 'EcogSPLang', 'EcogSPVisspat', 'EcogSPPlan', 'EcogSPOrgan', 'EcogSPDivatt', 'EcogSPTotal']
columns_neurobat         = ['LDELTOTAL', 'DIGITSCOR', 'TRABSCOR']
columns_pacc             = ['MMSE']
columns_pibpetsurv       = ['ACG', 'FRC', 'LTC', 'PAR', 'PRC']
columns_ptdemog          = ['PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY']
columns_ucsffsx7         = ['IMAGEUID', 'ST15CV', 'ST74CV', 'ST29SV', 'ST88SV', 'ST125SV', 'ST24CV', 'ST83CV', 'ST26CV', 'ST85CV', 'ST44CV', 'ST103CV', 'ST10CV']
columns_upennbiomkmaster = ['ABETA', 'TAU', 'PTAU']

In [355]:
def merge_table(base, other, columns, keys=['RID', 'VISCODE2']):
    cols_to_use = list(set(keys + columns))
    other_subset = other[cols_to_use].copy()
    return base.merge(other_subset, on=keys, how='left')

In [356]:
df = table_dxsum.copy()
df.rename(columns={'SUBJID': 'RID'}, inplace=True)

# ADSL
df = merge_table(df, table_adas, columns_adas)
df.rename(columns={'TOTSCORE': 'ADAS11'}, inplace=True)

# PTDEMOG
df = merge_table(df, table_ptdemog, columns_ptdemog)

# APOERES
df = merge_table(df, table_apoeres, columns_apores, keys=['RID'])
df['GENOTYPE'] = df['GENOTYPE'].astype(str).str.count('4')
df.rename(columns={'GENOTYPE': 'APOE4'}, inplace=True)

# ECOGPT
df = merge_table(df, table_ecogpt, columns_ecogpt)

# ECOGSP
df = merge_table(df, table_ecogsp, columns_ecogsp)

# NEUROBAT
df = merge_table(df, table_neurobat, columns_neurobat)

# PACC
df = merge_table(df, table_pacc, columns_pacc, keys=['RID', 'VISCODE', 'COLPROT'])

# PIBPETSUVR
df = merge_table(df, table_pibpetsurv, columns_pibpetsurv, keys=['RID', 'VISCODE', 'ORIGPROT'])
df['PIB'] = (df['ACG'] + df['FRC'] + df['LTC'] + df['PAR'] + df['PRC']) / 5
df.drop(columns=['ACG', 'FRC', 'LTC', 'PAR', 'PRC'], axis=1, inplace=True)

# UCSFFSX7
df = merge_table(df, table_ucsffsx7, columns_ucsffsx7)
df['Ventricles'] = df['ST15CV'] + df['ST74CV']
df['Hippocampus'] = df['ST29SV'] + df['ST88SV']
df['WholeBrain'] = df['ST125SV']
df['Entorhinal'] = df['ST24CV'] + df['ST83CV']
df['Fusiform'] = df['ST26CV'] + df['ST85CV']
df['MidTemp'] = df['ST44CV'] + df['ST103CV']
df['ICV'] = df['ST10CV']
df.drop(columns=['ST15CV', 'ST74CV', 'ST29SV', 'ST88SV', 'ST125SV', 'ST24CV', 'ST83CV', 'ST26CV', 'ST85CV', 'ST44CV', 'ST103CV', 'ST10CV'], axis=1, inplace=True)

# UPENNBIOMK
biomarker_df = table_upennbiomkmaster.drop_duplicates(subset=['RID', 'VISCODE'], keep='last')
df = merge_table(df, biomarker_df, columns_upennbiomkmaster, keys=['RID', 'VISCODE'])

# baseline values
cols_to_bl = ['RID', 'ADAS11', 'MMSE', 'LDELTOTAL', 'DIGITSCOR', 'TRABSCOR', 'IMAGEUID', 'Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV', 'EcogPtMem', 'EcogPtLang', 'EcogPtVisspat', 'EcogPtPlan', 'EcogPtOrgan', 'EcogPtDivatt', 'EcogPtTotal', 'EcogSPMem', 'EcogSPLang', 'EcogSPVisspat', 'EcogSPPlan', 'EcogSPOrgan', 'EcogSPDivatt', 'EcogSPTotal', 'ABETA', 'TAU', 'PTAU']
df_bl = df[df['VISCODE'] == 'bl'][cols_to_bl].copy()
df_bl = df_bl.rename(columns={c: c + '_bl' for c in cols_to_bl if c != 'RID'})
df = df.merge(df_bl, on='RID', how='left')

In [361]:
df.shape

(16645, 109)

In [363]:
df.to_csv('./master.csv')